In [ ]:
using Distributed

# if nprocs() == 1
#     addprocs(12)
# end

@everywhere using LatticeDecoder

@everywhere function random_bitstring!(b::Vector{Int64}, n)
    @inbounds for i = 1:n
        b[i] = rand(0:1)
    end
end

@everywhere function legacy_lsd_variable_node_message!(cn_message, vn, nb_idx::Int)
    msg_vector = LatticeDecoder._collect_msg_vector(vn, nb_idx)
    if length(msg_vector) == 1
        cn_message.mean = msg_vector[1].mean
        cn_message.var = vn.message.var
        cn_message.period = msg_vector[1].period
        return 1
    end

    lsd_inputs = LatticeDecoder.ListSphereDecodingInput(msg_vector)
    L, D = LatticeDecoder.simplified_lsd_legacy(lsd_inputs)
    isempty(D) && return 0

    candidate_gaussians = LatticeDecoder._calculate_candidate_gaussians(lsd_inputs, L, D, msg_vector)
    LatticeDecoder.moment_matching!(cn_message, candidate_gaussians)
    return length(D)
end

@everywhere function legacy_lsd_variable_node_messages!(tg, vn_idx::Int64)
    vn = tg.var_nodes[vn_idx]
    for j = 1:length(vn.neighbours)
        cn_idx, _ = vn.neighbours[j]
        idx = vn.pos_in_check_neighbour[j]
        cn = tg.check_nodes[cn_idx]
        legacy_lsd_variable_node_message!(cn.messages[idx], vn, j)
    end
    return nothing
end

@everywhere function legacy_lsd_variable_node_decision!(tg, vn_idx::Int64)
    vn = tg.var_nodes[vn_idx]
    msg_vector = LatticeDecoder._collect_msg_vector(vn)

    if length(msg_vector) == 2
        for msg in msg_vector
            if msg.var ≈ LatticeDecoder.MIN_VAR
                tg.bp_result[vn_idx] = msg.mean
                return 1
            end
        end
    end

    lsd_inputs = LatticeDecoder.ListSphereDecodingInput(msg_vector)
    L, D = LatticeDecoder.simplified_lsd_legacy(lsd_inputs)
    if !isempty(D)
        candidate_gaussians = LatticeDecoder._calculate_candidate_gaussians(lsd_inputs, L, D, msg_vector)
        LatticeDecoder.moment_matching!(vn.message, candidate_gaussians)
    end

    tg.bp_result[vn_idx] = vn.message.mean
    return length(D)
end

@everywhere function legacy_decision_step_lsd!(tg)
    for vn_idx = 1:tg.nv
        legacy_lsd_variable_node_decision!(tg, vn_idx)
    end
    return nothing
end

@everywhere function run_belief_propagation_lsd_legacy!(tg, message, σ, max_iter; search_interval::Float64=1.5)
    tg.search_interval = search_interval
    LatticeDecoder.initialize_messages!(tg, message, σ)
    for _ = 1:max_iter
        LatticeDecoder.check_node_iterations!(tg)
        for vn_idx = 1:tg.nv
            legacy_lsd_variable_node_messages!(tg, vn_idx)
        end
    end
    legacy_decision_step_lsd!(tg)
    return tg.bp_result
end

@everywhere function run_ldlc_decoder!(tg, y, σ, max_iter, decoder)
    if decoder == :paper_lsd
        ldlc_decoder = LDLCDecoder(
            tg;
            schedule = :parallel,
            algorithm = :lsd,
            sigma = σ,
            max_iterations = max_iter,
        )
        return run_decoder!(ldlc_decoder, y)
    elseif decoder == :legacy_lsd
        return run_belief_propagation_lsd_legacy!(tg, y, σ, max_iter)
    else
        throw(ArgumentError("decoder must be :paper_lsd or :legacy_lsd"))
    end
end

@everywhere function random_encoding_experiment(H, σ, max_iter, samples, decoder)
    n = size(H, 1)
    b = zeros(Int64, n)
    G = generator_matrix(H)
    tg = initialize_tanner_graph(H)
    errors = @distributed (+) for _ = 1:samples
        random_bitstring!(b, n)
        y = encode(b, G)
        y .+= sample_error(σ, n)
        bp_result = run_ldlc_decoder!(tg, y, σ, max_iter, decoder)
        dec = hard_decision(bp_result, H)
        count_symbol_errors(dec, b)
    end
    return errors / samples / n
end

"""
    agresti_coull_confidence_interval(p, n, z=1.96)

Compute the Agresti-Coull confidence interval for a binomial distribution.
The default value of z is for a 95% confidence interval.
To get the 99% confidence interval, use z=2.576.
To get the 99.9% confidence interval, use z=3.291.
"""
function agresti_coull_confidence_interval(p, n, z=1.96)
    n_tilde = n + z^2
    p_tilde = (p * n + z^2 / 2) / n_tilde
    return z * sqrt(p_tilde * (1 - p_tilde) / n_tilde)
end



In [ ]:
using Plots

samples = 1500;
max_iter = 25;
σ_capacity = lattice_capacity_std();
sigmas = range(σ_capacity, 0.8 * σ_capacity, 6);

d = 5;
ns = [128, 256, 512];
decoders = [
    (:paper_lsd, "paper/default LSD", :circle, :solid),
    (:legacy_lsd, "legacy capped LSD", :diamond, :dash),
];

p = plot(
    xlabel="σ (dB) from Capacity",
    ylabel="SER",
    title="Classical LDLC decoding: paper/default vs legacy LSD",
    yscale=:log10,
    grid=true,
)

for n in ns
    H = classical_ldlc(d, n, true)
    for (decoder, decoder_label, marker, linestyle) in decoders
        ber = [random_encoding_experiment(H, σ, max_iter, samples, decoder) for σ in sigmas]
        ribbon = agresti_coull_confidence_interval.(ber, samples * n)
        plot!(
            p,
            snr_db.(sigmas),
            ber;
            label="[$(n), $(d)] $(decoder_label)",
            lw=2,
            marker=marker,
            markersize=5,
            linestyle=linestyle,
            ribbon=ribbon,
        )
    end
end

display(p)


In [ ]:
# Quick single-instance comparison for one LDLC code and one noise sample.
d = 5
n = 128
σ = lattice_capacity_std()
max_iter = 25

H = classical_ldlc(d, n, true)
G = generator_matrix(H)
b = zeros(Int64, n)
random_bitstring!(b, n)
y = encode(b, G) .+ sample_error(σ, n)

paper_result = run_ldlc_decoder!(initialize_tanner_graph(H), copy(y), σ, max_iter, :paper_lsd)
legacy_result = run_ldlc_decoder!(initialize_tanner_graph(H), copy(y), σ, max_iter, :legacy_lsd)

paper_dec = hard_decision(paper_result, H)
legacy_dec = hard_decision(legacy_result, H)

println("paper/default symbol errors: ", count_symbol_errors(paper_dec, b))
println("legacy symbol errors:        ", count_symbol_errors(legacy_dec, b))
println("max |paper - legacy|:       ", maximum(abs.(paper_result .- legacy_result)))
println("hard decisions match:       ", all(paper_dec .== legacy_dec))
